# Week 06 — Validation Audit

**Model:** Week-5 text-classification / resume-screening model

This notebook practices rigorous validation, leakage detection, failure analysis, and evidence-aligned claims. It intentionally does **not** invent model results when the original dataset is unavailable.

## 1. Two research-paper findings + my methodology questions

### Finding 1 — Reported model performance
**Methodology question:** Where do the labels come from? I would want to know whether labels were independently annotated, taken from an existing system, or derived from a proxy. If labels are noisy or related to the model inputs, the measured performance could be optimistic.

**Constructive review:** The finding can still be useful, but clearly describing label provenance would help readers judge how closely the evaluation target represents the real outcome.

### Finding 2 — Validation performance
**Methodology question:** Does the validation design prevent related observations from appearing in both training and validation? If the same client, candidate, organization, document family, or time period can occur on both sides, the measured score may benefit from information overlap.

**Constructive review:** A genuine grouped-by-client or time-aware split would make the claim more demanding and test generalization more realistically.

These are methodology questions, not accusations. The purpose is to practice careful review constructively.

## 2. My model under an honest split — before / after

The original random split is the **before** measurement. For the **after** measurement, use a genuine client/candidate/document-family identifier or a genuine event timestamp.

Do **not** manufacture a group ID from the target label or row number. If no legitimate grouping/time field exists, report that limitation instead of fabricating an honest split.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

TARGET="target"       # change to the Week-5 target column
TEXT_COL="text"       # change to the Week-5 text column
GROUP_COL=None        # e.g. client_id / candidate_id if genuinely present
TIME_COL=None         # e.g. timestamp if genuinely present
RANDOM_STATE=42

# Attach the REAL Week-5 dataframe here:
# df=pd.read_csv("../data/your_dataset.csv")
df=None
print("Configuration loaded. Real Week-5 data must be attached before empirical results are claimed.")

In [ ]:
def random_baseline(df):
    Xtr,Xte,ytr,yte=train_test_split(
        df[TEXT_COL].fillna(""),df[TARGET],test_size=.20,
        random_state=RANDOM_STATE,stratify=df[TARGET])
    vec=TfidfVectorizer(max_features=5000)
    Xtr=vec.fit_transform(Xtr); Xte=vec.transform(Xte)
    model=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
    model.fit(Xtr,ytr); pred=model.predict(Xte)
    return accuracy_score(yte,pred),f1_score(yte,pred,average="macro"),yte,pred,Xte

if df is not None:
    before=random_baseline(df)
    print("BEFORE random split — accuracy:",before[0],"macro F1:",before[1])
else:
    before=None
    print("BEFORE: not executed because the real dataset is not attached.")

In [ ]:
def grouped_audit(df):
    if not GROUP_COL or GROUP_COL not in df.columns:
        raise ValueError("No genuine grouping field exists/configured. Do not invent one.")
    splitter=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=RANDOM_STATE)
    tr,te=next(splitter.split(df,groups=df[GROUP_COL]))
    train,test=df.iloc[tr],df.iloc[te]
    vec=TfidfVectorizer(max_features=5000)
    Xtr=vec.fit_transform(train[TEXT_COL].fillna(""))
    Xte=vec.transform(test[TEXT_COL].fillna(""))
    model=RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1)
    model.fit(Xtr,train[TARGET]); pred=model.predict(Xte)
    return accuracy_score(test[TARGET],pred),f1_score(test[TARGET],pred,average="macro"),test[TARGET],pred

if df is not None and GROUP_COL:
    after=grouped_audit(df)
    print("AFTER grouped split — accuracy:",after[0],"macro F1:",after[1])
else:
    after=None
    print("AFTER: no genuine group field configured; no result is claimed.")

### Interpretation

Use this safe wording after obtaining real results:

> Under the conventional random split, the model measured **[X]**. Under the stricter grouped/time-aware split, it measured **[Y]**. This is an observed validation difference under a more demanding split and provides directional evidence about generalization. It is not proof of deployment performance.

## 3. Leakage audit

I checked for:

- duplicate or near-duplicate documents across splits;
- client/candidate/document-family overlap;
- target-derived features;
- information that would only be known after the outcome;
- preprocessing fitted before splitting;
- TF-IDF vocabulary/statistics fitted on validation data;
- post-outcome HR information;
- train/test contamination.

For this text model, TF-IDF must be fitted **only on training text** and then used to transform held-out text.

In [ ]:
if df is not None:
    print("Duplicate rows:", int(df.duplicated().sum()))
    if TEXT_COL in df.columns:
        print("Duplicate text values:", int(df[TEXT_COL].fillna("").astype(str).duplicated().sum()))
else:
    print("Attach the real dataset to execute leakage checks.")

## 4. Real failure examples

The audit should report actual held-out mistakes rather than only aggregate metrics.

| Example | Actual | Predicted | What happened? | Likely reason |
|---|---|---|---|---|
| 1 | — | — | — | — |
| 2 | — | — | — | — |
| 3 | — | — | — | — |

Questions for each error: Was the resume ambiguous? Was text extraction incomplete? Did categories share terminology? Was important information missing? Is the issue data quality, labeling, or model limitation?

In [ ]:
def failure_table(texts,actual,predicted,n=5):
    out=pd.DataFrame({"text":list(texts),"actual":list(actual),"predicted":list(predicted)})
    return out[out.actual!=out.predicted].head(n)

print("Run failure_table() with the real held-out texts, labels, and predictions.")

## 5. Claim rewrite

### Too strong
> “The model accurately screens resumes and can reliably automate HR candidate selection.”

### Evidence-aligned
> “The model measured classification performance on the evaluated resume dataset and produced ranked category predictions. The results provide directional evidence that the approach can support resume categorization, but they do not establish reliable automated hiring decisions or deployment performance.”

### Public-safe vocabulary
Use **observed**, **measured**, **directional**, and **decision-support**. Avoid claims such as *guarantees*, *fully automates hiring*, *works reliably in production*, or *generalizes to all resumes*.

## Self-check

- [x] Two findings have constructive methodology questions.
- [x] Random validation is treated as a baseline.
- [x] Grouped validation requires a genuine grouping field.
- [x] TF-IDF is fitted inside the training split.
- [x] Leakage checks are defined.
- [x] Failure analysis is defined.
- [x] Claims are rewritten conservatively.
- [ ] Real Week-5 dataset attached.
- [ ] Actual before/after metrics executed.
- [ ] Actual failure examples inserted.

**Audit status:** methodology complete; empirical results must come from the real Week-5 data.